# Experiment C: census of privileged control in 25 real tokenised-asset contracts

Reads, for the 25 contracts fixed in `docs/data/exploratory/census_tokens.txt`, exactly the observations listed in `docs/EXPLORATORY_PLAN_C_ADDENDUM.md` (tag v1.6-census-plan): source verification and ABI, proxy slots, `owner()` / `defaultAdmin()`, and for each controlling address whether it is an account, a multisig or a timelock. It does **not** classify or analyse anything: that runs later on GitHub from the committed data.

**How to use:** menu *Runtime, Run all*. When asked, paste a **new** Etherscan API key (hidden; never printed or saved). It takes a few minutes. At the end `exploratory_C_data.zip` downloads; send it back, then delete the key on etherscan.io/apidashboard.


In [ ]:
# Cell 2: functions (nothing to edit)
from __future__ import annotations

import datetime as dt, hashlib, json, os, re, subprocess, sys, time, zipfile
from pathlib import Path

REPO_URL = "https://github.com/FahdSaif/RWA-Asset-tokenization.git"
REPO_REF = "v1.6-census-plan"                           # the tag that fixes the sample list and the rules (docs/EXPLORATORY_PLAN_C_ADDENDUM.md)
CHAINID = 1
BASE = "https://api.etherscan.io/v2/api"
ZERO = "0x" + "0" * 40
ADMIN_SLOT = "0xb53127684a568b3173ae13b9f8a6016e243e63b6e8ee1178d6a717850b5d6103"      # EIP-1967 admin
IMPL_SLOT = "0x360894a13ba1a3210667c828492db98dca3e2076cc3735a920a3ca505d382bbc"       # EIP-1967 implementation
SEL = dict(owner="0x8da5cb5b", threshold="0xe75235b8", owners="0xa0e67e2b", min_delay="0xf27a0c92", default_admin="0x84ef8ffc")
OUT = Path("exploratory_C")


def url(params: dict, key: str) -> str:
    q = "&".join(f"{k}={v}" for k, v in {"chainid": CHAINID, **params}.items())
    return f"{BASE}?{q}&apikey={key}"


class Reader:
    """Thin reader around the Etherscan API; every call is rate-limited; errors are collected, never raised for one contract."""

    def __init__(self, get_json, key: str, pause: float = 0.3):
        self.get_json, self.key, self.pause = get_json, key, pause
        self.errors: list[str] = []

    def _call(self, params: dict):
        time.sleep(self.pause)
        return self.get_json(url(params, self.key))

    def result(self, params: dict, what: str, soft: bool = False):
        """soft=True: a failure is an expected 'no answer' (for example owner() on a contract that has no owner()) and is not an error."""
        def fail(msg):
            if not soft:
                self.errors.append(msg)
            return None
        try:
            d = self._call(params)
        except Exception as e:                                              # network or parse problem: record it and go on
            return fail(f"{what}: {type(e).__name__}")
        if isinstance(d, dict) and "error" in d:
            return fail(f"{what}: {str(d['error'])[:80]}")
        if isinstance(d, dict) and d.get("status") == "0" and not isinstance(d.get("result"), (list, dict)):
            return fail(f"{what}: {str(d.get('message'))[:40]} {str(d.get('result'))[:60]}")
        return d.get("result") if isinstance(d, dict) else None

    def source(self, address: str):
        r = self.result(dict(module="contract", action="getsourcecode", address=address), f"getsourcecode {address[:8]}")
        return r[0] if isinstance(r, list) and r else None

    def storage(self, address: str, slot: str):
        r = self.result(dict(module="proxy", action="eth_getStorageAt", address=address, position=slot, tag="latest"), f"storage {address[:8]}")
        return addr_from_word(r)

    def call(self, address: str, data: str):
        r = self.result(dict(module="proxy", action="eth_call", to=address, data=data, tag="latest"), f"call {data[:10]} {address[:8]}", soft=True)
        return r if isinstance(r, str) and r.startswith("0x") and len(r) > 2 else None

    def code(self, address: str):
        r = self.result(dict(module="proxy", action="eth_getCode", address=address, tag="latest"), f"code {address[:8]}")
        return None if r is None else (r not in ("0x", "0x0"))


def addr_from_word(word):
    """Last 20 bytes of a 32-byte word as an address; None if the word is missing or not a word; the zero address stays the zero address."""
    if not isinstance(word, str) or not re.fullmatch(r"0x[0-9a-fA-F]{64}", word):
        return None
    return "0x" + word[-40:].lower()


def uint_from_word(word):
    return int(word, 16) if isinstance(word, str) and re.fullmatch(r"0x[0-9a-fA-F]{64}", word) else None


def array_len(data):
    """Length of an ABI-encoded address[] (offset word, length word, items)."""
    if not isinstance(data, str) or len(data) < 2 + 128:
        return None
    return int(data[2 + 64:2 + 128], 16)


def abi_functions(abi_text) -> list[str]:
    try:
        return [e["name"] for e in json.loads(abi_text) if e.get("type") == "function" and "name" in e]
    except Exception:
        return []


def observe_controller(rd: Reader, address: str) -> dict:
    info = dict(is_contract=rd.code(address))
    if info["is_contract"]:
        info["threshold"] = uint_from_word(rd.call(address, SEL["threshold"]))
        info["n_owners"] = array_len(rd.call(address, SEL["owners"]))
        info["min_delay"] = uint_from_word(rd.call(address, SEL["min_delay"]))
        info["owner"] = addr_from_word(rd.call(address, SEL["owner"]))
    return info


def observe_token(rd: Reader, address: str, rank: int) -> dict:
    rd.errors = []
    rec = dict(address=address, rank=rank, contract_name=None, source_verified=False, is_proxy=False, implementation=None,
               abi_functions=[], owner=None, default_admin=None, admin_slot=None, controllers={})
    src = rd.source(address)
    funcs = []
    if src:
        rec["contract_name"] = src.get("ContractName") or None
        rec["source_verified"] = bool(src.get("SourceCode")) and "not verified" not in str(src.get("ABI", "")).lower()
        funcs += abi_functions(src.get("ABI"))
        if src.get("Proxy") == "1" and re.fullmatch(r"0x[0-9a-fA-F]{40}", str(src.get("Implementation", ""))):
            rec["implementation"] = src["Implementation"].lower()
    impl_slot = rd.storage(address, IMPL_SLOT)
    if impl_slot and impl_slot != ZERO:
        rec["implementation"] = rec["implementation"] or impl_slot
    rec["is_proxy"] = bool(rec["implementation"])
    if rec["implementation"]:
        isrc = rd.source(rec["implementation"])
        if isrc:
            funcs += abi_functions(isrc.get("ABI"))
    rec["abi_functions"] = sorted(set(funcs))
    rec["admin_slot"] = rd.storage(address, ADMIN_SLOT)
    rec["owner"] = addr_from_word(rd.call(address, SEL["owner"]))
    rec["default_admin"] = addr_from_word(rd.call(address, SEL["default_admin"]))
    todo = [a for a in (rec["admin_slot"], rec["owner"], rec["default_admin"]) if a and a != ZERO]
    seen = set()
    while todo:                                                              # the controllers, plus one further hop through a contract owner
        a = todo.pop()
        if a in seen:
            continue
        seen.add(a)
        rec["controllers"][a] = observe_controller(rd, a)
        o = rec["controllers"][a].get("owner")
        if o and o != ZERO and o not in seen and len(seen) < 8:
            todo.append(o)
    rec["errors"] = list(rd.errors)
    return rec


def sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


def read_tokens(path: str) -> list[tuple[str, int]]:
    out = []
    for ln in open(path):
        if ln.strip() and not ln.startswith("#"):
            parts = ln.split()
            out.append((parts[0].lower(), int(parts[1])))
    return out


def build(records: list[dict], repo_commit: str) -> dict:
    OUT.mkdir(exist_ok=True)
    (OUT / "census_raw.json").write_text(json.dumps(dict(tokens=records), indent=1))
    summary = dict(plan_tag=REPO_REF, repo_commit=repo_commit, retrieved_utc=dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                   tokens=len(records), with_errors=sum(bool(r["errors"]) for r in records), source_verified=sum(r["source_verified"] for r in records),
                   proxies=sum(r["is_proxy"] for r in records), sha256={"census_raw.json": sha256(OUT / "census_raw.json")})
    (OUT / "census_summary.json").write_text(json.dumps(summary, indent=1))
    with zipfile.ZipFile("exploratory_C_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.glob("*")):
            z.write(p, p.name)
    return summary


In [ ]:
# Cell 3: run
import getpass, requests
from google.colab import files

repo = os.environ.get("REPO_LOCAL", "repo")
if not os.path.exists(repo):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, repo], check=True)
commit = subprocess.run(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
TOKENS = read_tokens(f"{repo}/rwa-threatlab/docs/data/exploratory/census_tokens.txt")
print(f"plan tag {REPO_REF}, commit {commit[:7]}: {len(TOKENS)} token contracts in the fixed list")
assert len(TOKENS) == 25, "the fixed list must have 25 contracts"

key = getpass.getpass("Paste your Etherscan API key (hidden): ").strip()

def get_json(u):
    try:
        r = requests.get(u, timeout=60)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:          # never show the URL: it contains the key
        raise RuntimeError(f"network error: {type(e).__name__}") from None

rd = Reader(get_json, key)
records = []
for address, rank in TOKENS:
    rec = observe_token(rd, address, rank)
    records.append(rec)
    print(f"{rank:2d} {address[:10]}... {str(rec['contract_name'])[:22]:22s} verified={rec['source_verified']!s:5} proxy={rec['is_proxy']!s:5} "
          f"controllers={len(rec['controllers'])} errors={len(rec['errors'])}")

summary = build(records, commit)
print(json.dumps({k: v for k, v in summary.items() if k != "sha256"}, indent=1))
if summary["with_errors"] > 10:
    print("\nWARNING: many contracts had read errors; send the printout to the author before using this data.")
files.download("exploratory_C_data.zip")
